In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# 加载环境变量 + 初始化客户端
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
base_url = "https://ark.cn-beijing.volces.com/api/v3"
client = OpenAI(
    api_key=api_key,
    base_url=base_url,
    timeout=30.0
)
print("客户端初始化成功")


客户端初始化成功


In [3]:
import json

def llm_chat_json(messages, model="ep-20260324112743-tglmk", max_retry=3):
    """调用模型强制返回JSON，解析失败自动重试（最多3次）"""
    for attempt in range(1, max_retry + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                response_format={"type": "json_object"},   # 强制JSON
                temperature=0.3
            )
            content = resp.choices[0].message.content
            data = json.loads(content)                     # 解析成Python字典
            return data, content                           # 成功就返回
        except json.JSONDecodeError as e:
            print(f"第{attempt}次：JSON解析失败，重试中... {e}")
        except Exception as e:
            print(f"第{attempt}次：调用异常，重试中... {e}")
    return None, None                                      # 3次都失败

# ===== 测试 =====
messages = [
    {"role": "system", "content": "你是天气助手，只输出JSON，不要任何解释"},
    {"role": "user", "content": "深圳今天天气怎么样？输出JSON，包含city、temperature、weather字段"}
]
data, raw = llm_chat_json(messages)
print("解析结果:", data)
print("数据类型:", type(data))


解析结果: {'city': '深圳', 'temperature': '25-32℃', 'weather': '多云'}
数据类型: <class 'dict'>


In [4]:
def llm_chat_json_schema(messages, model="ep-20260324112743-tglmk", max_retry=3):
    """调用模型，按JSON Schema强制输出"""
    json_schema = {
        "name": "weather_response",
        "strict": True,                                  # 严格模式
        "schema": {
            "type": "object",
            "required": ["city", "temperature", "weather"],
            "properties": {
                "city": {"type": "string", "description": "城市名称"},
                "temperature": {"type": "number", "description": "气温数值（摄氏度）"},
                "weather": {"type": "string", "enum": ["晴", "多云", "雨", "雪", "阴"],
                            "description": "天气状况"}
            },
            "additionalProperties": False                # 不许多出字段
        }
    }
    for attempt in range(1, max_retry + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                response_format={"type": "json_schema", "json_schema": json_schema},
                temperature=0.3
            )
            content = resp.choices[0].message.content
            data = json.loads(content)
            return data, content
        except json.JSONDecodeError as e:
            print(f"第{attempt}次：JSON解析失败，重试中... {e}")
        except Exception as e:
            print(f"第{attempt}次：调用异常，重试中... {e}")
    return None, None

# ===== 测试 =====
messages = [
    {"role": "system", "content": "你是天气助手，按给定格式输出"},
    {"role": "user", "content": "深圳今天天气怎么样？"}
]
data, raw = llm_chat_json_schema(messages)
print("解析结果:", data)
print("数据类型:", type(data))


解析结果: {'city': '深圳', 'temperature': 26, 'weather': '晴'}
数据类型: <class 'dict'>


In [5]:
# ===== 函数调用实战：给模型函数简历 =====
functions = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询指定城市的实时天气",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "城市名称"}
            },
            "required": ["city"]
        }
    }
}]

resp = client.chat.completions.create(
    model="ep-20260324112743-tglmk",
    messages=[
        {"role": "system", "content": "你是天气助手，需要时调用工具获取天气"},
        {"role": "user", "content": "北京今天天气怎么样？"}
    ],
    tools=functions                       # 把函数简历给模型
)

msg = resp.choices[0].message
print("finish_reason:", resp.choices[0].finish_reason)
if msg.tool_calls:
    for tc in msg.tool_calls:
        print("函数名:", tc.function.name)
        print("参数:", tc.function.arguments)     # 注意：这是JSON字符串
else:
    print("模型没调用函数，直接回答:", msg.content)


finish_reason: tool_calls
函数名: get_weather
参数:  {"city": "北京"}


In [6]:
import json

# ===== 程序端的"真函数"（本地模拟，以后换成真API） =====
def get_weather(city):
    """本地模拟天气查询，真实场景替换成调用天气API"""
    fake_data = {
        "深圳": {"temperature": 28, "weather": "晴"},
        "北京": {"temperature": 12, "weather": "多云"},
        "上海": {"temperature": 22, "weather": "阴"}
    }
    return fake_data.get(city, {"temperature": "未知", "weather": "未知"})

# ===== 完整闭环 =====
# ① 用户提问 → 模型输出调用指令
messages = [
    {"role": "system", "content": "你是天气助手，需要时调用工具获取天气"},
    {"role": "user", "content": "上海今天天气怎么样？"}
]
resp = client.chat.completions.create(
    model="ep-20260324112743-tglmk",
    messages=messages,
    tools=functions
)
msg = resp.choices[0].message
print("① 模型输出指令:", msg.tool_calls[0].function.name, msg.tool_calls[0].function.arguments)
messages.append(msg)                          # 模型回复加入对话

# ② 程序执行函数
if msg.tool_calls:
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)   # 解析参数
        result = get_weather(args["city"])         # 程序真正执行！
        print("② 程序执行函数 →", result)
        messages.append({                          # 结果回填
            "role": "tool",
            "tool_call_id": tc.id,                 # 关联到那次调用
            "content": json.dumps(result)          # 结果转JSON字符串
        })

# ③ 结果给模型 → 生成最终回答
resp2 = client.chat.completions.create(
    model="ep-20260324112743-tglmk",
    messages=messages
)
print("③ 最终回答:", resp2.choices[0].message.content)


① 模型输出指令: get_weather  {"city": "上海"}
② 程序执行函数 → {'temperature': 22, 'weather': '阴'}
③ 最终回答: 上海今天天气晴朗，气温22摄氏度，体感舒适~


In [7]:
import json

# ===== 程序端的"真函数"（本地模拟，以后换成真API） =====
def get_weather(city):
    """本地模拟天气查询，真实场景替换成调用天气API"""
    fake_data = {
        "深圳": {"temperature": 28, "weather": "晴"},
        "北京": {"temperature": 12, "weather": "多云"},
        "上海": {"temperature": 22, "weather": "阴"}
    }
    return fake_data.get(city, {"temperature": "未知", "weather": "未知"})

# ===== 完整闭环 =====
# ① 用户提问 → 模型输出调用指令
messages = [
    {"role": "system", "content": "你是天气助手，需要时调用工具获取天气"},
    {"role": "user", "content": "上海今天天气怎么样？"}
]
resp = client.chat.completions.create(
    model="ep-20260324112743-tglmk",
    messages=messages,
    tools=functions
)
msg = resp.choices[0].message
print("① 模型输出指令:", msg.tool_calls[0].function.name, msg.tool_calls[0].function.arguments)
messages.append(msg)                          # 模型回复加入对话

# ② 程序执行函数
if msg.tool_calls:
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)   # 解析参数
        result = get_weather(args["city"])         # 程序真正执行！
        print("② 程序执行函数 →", result)
        messages.append({                          # 结果回填
            "role": "tool",
            "tool_call_id": tc.id,                 # 关联到那次调用
            "content": json.dumps(result)          # 结果转JSON字符串
        })

# ③ 结果给模型 → 生成最终回答
resp2 = client.chat.completions.create(
    model="ep-20260324112743-tglmk",
    messages=messages
)
print("③ 最终回答:", resp2.choices[0].message.content)


① 模型输出指令: get_weather  {"city": "上海"}
② 程序执行函数 → {'temperature': 22, 'weather': '阴'}
③ 最终回答: 上海今天的天气是晴，温度为22度。
